## Executive Summary — Accessing Claude with the API

- Request lifecycle: client → **your server** → Anthropic API → Claude → response back. Never call the API directly from client-side code (exposes your secret key).
- Every request needs: API key, `model`, `messages`, `max_tokens`.
- Get a key at console.anthropic.com → "Get API Keys" → "Create Key" (copy immediately, shown once).
- Use `client.messages.create()`; text lives at `message.content[0].text`.
- Claude is **stateless** — you must resend the full conversation every time. Build this with helper functions: `add_user_message()`, `add_assistant_message()`, `chat()`.
- `system` param shapes Claude's role/behavior; never pass `system=None` — only include the key conditionally.
- `temperature` (0-1) controls randomness: low → factual/code, high → creative/brainstorming.
- `stream=True` (or `client.messages.stream()`) streams text chunk-by-chunk for better UX; `stream.get_final_message()` still gives the full object.
- ★ To get **clean structured output** (JSON/code, no commentary): prefill the assistant turn (e.g. ` ```json `) + `stop_sequences=["```"]`.

# Accessing Claude with the API

### Accessing the API
- Five-step request flow: client → your server → Anthropic API → Claude processing → response back down to client.
- Never call the API directly from client-side code — exposes your secret key.
- Every request must include: API key, `model`, `messages`, `max_tokens`.
- Inside Claude: tokenization → embedding → contextualization → generation.
- Generation stops when: max tokens reached, natural end-of-sequence token, or a stop sequence is hit.
- API response contains: message text, `usage` (token counts), `stop_reason`.

### Getting an API key
1. Go to console.anthropic.com and log in.
2. Click **Get API Keys** (top right of dashboard).
3. Click **Create Key**.
4. Choose workspace "Default", give the key a name.
5. Copy the key immediately — it's shown only once. If you lose it, delete and regenerate.

### Making a request
- Install deps: `%pip install anthropic python-dotenv`
- Store the key in a `.env` file, never hardcode it. Add `.env` to `.gitignore`.
- `client.messages.create()` needs `model`, `max_tokens` (a ceiling, not a target — Claude stops early if done), `messages`.
- Get text back via `message.content[0].text`.

In [ ]:
# Setup (repeated at the top of nearly every lesson from here on)
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

# Making a first request
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "What is quantum computing? Answer in one sentence"}
    ]
)
print(message.content[0].text)


### ★ Multi-Turn conversations
- Claude is **stateless** — no memory between requests. You must resend the entire message history each call.
- Flow: send user msg → append Claude's reply as an assistant message → append the next user msg → resend everything.
- These three helper functions are reused for the rest of the course.

In [ ]:
# FUNCTION - adds a user turn to the running message list
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# FUNCTION - adds an assistant turn to the running message list
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# FUNCTION - sends the full message history to Claude and returns just the text
def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# Usage: build up context turn by turn
messages = []
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
add_assistant_message(messages, answer)
add_user_message(messages, "Write another sentence")
final_answer = chat(messages)


### System prompts
- `system` shapes Claude's tone/role/behavior (e.g. a patient tutor who won't just hand out answers).
- Passed as a plain string via the `system=` param.
- Claude's API rejects `system=None` — only include the `system` key in params when one is actually provided.

In [ ]:
# FUNCTION - chat() updated to optionally accept a system prompt
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)


### Temperature
Decimal 0–1 controlling randomness of next-token sampling.

| Range | Use for |
|---|---|
| 0.0 - 0.3 | Factual answers, coding, data extraction, moderation |
| 0.4 - 0.7 | Summarization, education, constrained creative writing |
| 0.8 - 1.0 | Brainstorming, creative writing, jokes, marketing copy |

Doesn't guarantee different output every time — it just changes the odds.

In [ ]:
# FUNCTION - chat() updated to accept a temperature
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text


### Response streaming
- Standard requests can leave users staring at a spinner for 10-30s. Streaming shows text as it's generated.
- Raw approach: `stream=True` on `.create()`, iterate over events (`MessageStart`, `ContentBlockStart`, `ContentBlockDelta` — the actual text chunks, `ContentBlockStop`, `MessageDelta`, `MessageStop`).
- Simpler: use the SDK's `client.messages.stream()` context manager + `stream.text_stream`.
- After streaming finishes, `stream.get_final_message()` still gives you the complete message object (e.g. for DB storage).

In [ ]:
# Simplified text streaming (preferred way)
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

    final_message = stream.get_final_message()


### ★ Structured data
- Claude likes to wrap JSON/code in markdown and add explanation — a problem when your app needs to copy/paste raw output directly.
- **Fix:** prefill the assistant turn with the opening marker (e.g. ` ```json `) and pass `stop_sequences=["```"]`. Claude continues as if it already started the block, and generation halts the instant it tries to close it.
- Works for any structured format Claude would naturally wrap: JSON, Python, regex, CSV, bulleted lists.
- Parse/clean the result as usual afterward, e.g. `json.loads(text.strip())`.

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

import json
clean_json = json.loads(text.strip())


## Executive Summary — Prompt Evaluation

- Writing a prompt isn't enough — you must **measure** how well it performs. Prompt engineering = writing better prompts; prompt evaluation = objectively scoring them.
- Shipping after one or two manual tests is a trap — real users hit edge cases you never imagined.
- Standard eval workflow (5 steps, repeat until happy): draft prompt → build eval dataset → run each case through Claude → grade each output 1–10 → tweak the prompt and rerun, comparing average scores.
- Datasets can be hand-written or Claude-generated (a fast/cheap model like Haiku is fine for this, since it's just test data).
- Three grader types: **code graders** (format/syntax checks, fast & objective), **model graders** (another Claude call judges quality/instruction-following, flexible), **human graders** (best for nuance, slow/expensive).
- ★ Model graders must be asked for strengths/weaknesses/reasoning alongside the score — a bare "score 1-10" request makes models default to a mediocre ~6.
- Code graders are ideal for objective checks: try to parse the output as JSON/Python/regex; return 10 on success, 0 on failure.
- Combine both: `score = (model_score + syntax_score) / 2`.

# Prompt Evaluation

### Prompt evaluation (intro)
- Prompt engineering = toolkit of techniques (multishot, XML tags, etc). Prompt evaluation = automated measurement of how well a prompt actually performs.
- Three paths after drafting a prompt:
  1. Ship after one test — risky, breaks on real users.
  2. Tweak for a couple of corner cases — better, still risky.
  3. ★ Run it through a full **evaluation pipeline** and iterate on objective scores — more upfront work/cost, far more reliability.
- Real users always surface edge cases you didn't anticipate in manual testing.

### ★ A typical eval workflow
Five-step loop, repeat steps 4-5 until satisfied:
1. **Draft a prompt** — a template with variables, e.g. `{question}`.
2. **Create an eval dataset** — sample inputs representing real use (by hand or Claude-generated).
3. **Feed through Claude** — run every dataset item through the prompt to get real outputs.
4. **Feed through a grader** — score each output 1-10.
5. **Change the prompt and repeat** — compare average scores numerically (e.g. 7.66 → 8.7) to confirm a real improvement, not just a different variation.

### Generating test datasets
- Example: building a prompt that outputs Python / JSON / regex for AWS tasks.
- Use a fast/cheap model (e.g. Haiku) to generate the dataset — it's just test data, not the real prompt under test.
- Save with `json.dump(dataset, f, indent=2)` into `dataset.json` for reuse across iterations.
- 📥 Downloadable notebook: `001_prompt_evals.ipynb`

In [ ]:
import json

# FUNCTION - generates an evaluation dataset of tasks by asking Claude for JSON
def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used
    to evaluate prompts that generate Python, JSON, or Regex specifically for
    AWS-related tasks. Generate an array of JSON objects, each representing a task
    that requires Python, JSON, or a Regex to complete.

    * Focus on tasks solvable with a single Python function, JSON object, or regex
    * Focus on tasks that do not require writing much code
    Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)


### Running the eval
Three core functions form the pipeline:
- FUNCTION - `run_prompt(test_case)`: merges the prompt template with one test case's task, sends to Claude, returns raw output.
- FUNCTION - `run_test_case(test_case)`: calls `run_prompt()`, then grades it (starts as a hardcoded placeholder score of 10 until grading is built).
- FUNCTION - `run_eval(dataset)`: loops `run_test_case()` over every item, collects results into a list.
- A full dataset run can take ~30s even on Haiku — expect this on first run.

In [ ]:
# FUNCTION - merges prompt template with one test case, returns Claude's raw output
def run_prompt(test_case):
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

# FUNCTION - runs one test case end-to-end and (for now) hardcodes the score
def run_test_case(test_case):
    output = run_prompt(test_case)
    score = 10  # TODO - real grading comes later
    return {"output": output, "test_case": test_case, "score": score}

# FUNCTION - runs every test case in the dataset and collects results
def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)


### ★ Model based grading
- A grader returns a measurable score (1-10). Three types: code, model, human graders.
  - **Code graders** — length/keyword checks, syntax validation, readability. Fast, objective, narrow.
  - **Model graders** — another Claude call assesses quality, instruction-following, completeness, helpfulness, safety. Flexible.
  - **Human graders** — best for nuance (comprehensiveness, tone), but slow/expensive.
- Define clear criteria first, e.g. for code generation: **Format** (no explanation), **Valid Syntax**, **Task Following**.
- ★ Always ask the grader for `strengths` / `weaknesses` / `reasoning` alongside the `score` — without this context, models default to a mediocre ~6.
- 📥 Downloadable notebook: `001_prompt_evals_grader.ipynb`

In [ ]:
# FUNCTION - uses a second Claude call to grade a solution against its task
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {test_case["task"]}
    Solution: {output}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

# FUNCTION - run_test_case updated to call the real grader
def run_test_case(test_case):
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    return {"output": output, "test_case": test_case, "score": score, "reasoning": reasoning}

from statistics import mean

# FUNCTION - run_eval updated to also print the average score across all cases
def run_eval(dataset):
    results = []
    for test_case in dataset:
        results.append(run_test_case(test_case))
    average_score = mean([r["score"] for r in results])
    print(f"Average score: {average_score}")
    return results


### Code based grading
- Complements the model grader by objectively checking **Format** and **Valid Syntax**.
- Each validator tries to parse the output as its target format; returns 10 on success, 0 on failure.
- Dataset items need a `"format"` field (`python` / `json` / `regex`) so the right validator is used.
- Make the prompt explicitly demand raw output only ("no comments or commentary"), and prefill with a generic ` ```code ` block to push Claude toward clean output.
- Combine scores: `score = (model_score + syntax_score) / 2` — equal weight by default, adjustable.
- 📥 Downloadable notebook: `001_prompt_evals_fns.ipynb`

In [ ]:
import ast, re, json

# FUNCTION - returns 10 if text parses as valid JSON, else 0
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

# FUNCTION - returns 10 if text parses as valid Python, else 0
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

# FUNCTION - returns 10 if text compiles as a valid regex, else 0
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

# Combine content-quality score with syntax-correctness score
model_grade = grade_by_model(test_case, output)
model_score = model_grade["score"]
syntax_score = validate_json(output)  # pick validator based on test_case["format"]
score = (model_score + syntax_score) / 2


## Executive Summary — Prompt Engineering Techniques

- Prompt engineering is an **iterative loop**: set a goal → write a deliberately basic initial prompt → evaluate → apply one technique → re-evaluate → repeat until scores stabilize.
- Track progress with an eval pipeline (e.g. a `PromptEvaluator` class) that auto-generates a small dataset (2-3 cases while iterating) and grades with a model, so every technique's impact is measurable.
- ★ **Being clear & direct**: lead with a direct action verb ("Generate...", not a vague question). This alone took one example from 2.32 → 3.92.
- ★ **Being specific**: add (a) output-quality guidelines (length, structure, required elements) and (b) step-by-step process instructions for complex/judgment-heavy tasks. Guidelines alone took the same example from 3.92 → 7.86 — always include output guidelines; add process steps only when the task needs multi-angle reasoning.
- **Structure with XML tags**: wrap distinct chunks of context (data, code, docs) in descriptive custom tags (`<athlete_information>`, `<my_code>`) so Claude doesn't conflate instructions with content — matters most with long or mixed-content prompts.
- **Providing examples** (one-shot/multi-shot): give sample input → ideal output pairs (in `<sample_input>`/`<ideal_output>` tags) to handle edge cases like sarcasm. Reuse your best real eval outputs (score 10) as examples, and briefly explain why each is good.

# Prompt Engineering Techniques

### Prompt engineering (overview)
- Iterative loop: set goal → write initial (deliberately naive) prompt → evaluate → apply a technique → re-evaluate → repeat.
- Use an eval pipeline class, e.g. `evaluator = PromptEvaluator(max_concurrent_tasks=5)` — start concurrency low (~3) to avoid rate limits.
- `evaluator.generate_dataset(task_description=..., prompt_inputs_spec={...}, output_file="dataset.json", num_cases=3)` — keep `num_cases` small (2-3) while iterating for speed; scale up for final validation.
- Expect a low baseline score from the naive prompt (e.g. 2.3/10 is typical for a first attempt — not a bug).
- `evaluator.run_evaluation(run_prompt_function=..., dataset_file=..., extra_criteria="...")` returns a score plus a detailed HTML report showing per-case reasoning.
- 📥 Downloadable notebooks: `001_prompting.ipynb`, `002_prompting_completed.ipynb`

In [ ]:
evaluator = PromptEvaluator(max_concurrent_tasks=5)

dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    output_file="dataset.json",
    num_cases=3
)

# FUNCTION - deliberately basic baseline prompt to establish a starting score
def run_prompt(prompt_inputs):
    prompt = f"""
    What should this person eat?
    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}
    """
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """
)


### ★ Being clear and direct
- The **first line** of the prompt matters most.
- *Clear* = simple, unambiguous language stating exactly what you want.
- *Direct* = an instruction, not a question — start with an action verb ("Write", "Create", "Generate", "Identify").
- Example: "What should this person eat?" → **"Generate a one-day meal plan for an athlete that meets their dietary restrictions."**
- Result: score jumped **2.32 → 3.92** from this rewrite alone.

### ★ Being specific
Two types of specificity, often combined:
1. **Output quality guidelines** — length, structure, required elements, tone. Use in almost every prompt.
2. **Process steps** — ordered instructions for Claude to follow. Best for complex/judgment-heavy tasks (troubleshooting, multi-factor analysis).

Example guideline list (meal plan prompt): daily calories, protein/fat/carb amounts, meal timing, restriction compliance, portions in grams, budget constraint.

Result: this one change raised the score **3.92 → 7.86**.

Rule of thumb: always add output guidelines; only add process steps when the task needs multi-angle reasoning.

### Structure with XML tags
- Wrap distinct content blocks in descriptive custom tags so Claude doesn't conflate instructions, data, and code.
- Use specific names, not generic ones: `<sales_records>` not `<data>`; `<athlete_information>`; `<my_code>` / `<docs>`.
- Most valuable with large or mixed context (long data dumps, code + docs together); less critical for short simple prompts.

### ★ Providing examples
- "One-shot" (1 example) / "multi-shot" (several examples): give sample input → ideal output pairs to *show*, not just tell, Claude what you want.
- Wrap examples in tags like `<sample_input>` / `<ideal_output>` for clarity.
- Best for: edge cases (e.g. sarcasm in sentiment analysis), exact output formats, tone/style matching, ambiguous-input handling.
- ★ Practical tip: mine your own eval results for the highest-scoring (e.g. 10/10) outputs and reuse them as examples.
- Don't just show the example — briefly explain *why* it's good; this extra reasoning context helps Claude generalize the pattern, not just copy the format.

## Executive Summary — Tool Use with Claude

- Tools let Claude access real-time info / external systems it wasn't trained on (weather, dates, DB lookups, reminders, etc).
- ★ The tool use loop: (1) write a plain Python tool function, (2) write a JSON schema describing it, (3) call Claude with the schema in `tools=[...]`, (4) run the tool Claude asks for, (5) send the result back and call Claude again.
- Claude signals it wants a tool via `response.stop_reason == "tool_use"` and returns a multi-block message (text block + `ToolUse` block with an `id`, `name`, `input`).
- Run the requested function, wrap the output in a `tool_result` block (matching `tool_use_id`), append it as a new user message, and call Claude again — loop until `stop_reason` is no longer `"tool_use"`.
- Claude can request several tools in one turn or across turns — route by `tool_name` in a single `run_tool()` dispatcher, and keep looping until Claude has everything it needs.
- Validate tool function inputs and raise clear errors — Claude reads error messages and often retries with corrected input.
- Two tools are **built into Claude** (schema only, you supply the implementation for the text editor; Claude runs the actual search for web search):
  - The **text editor tool** — view/edit/create files, needs a small version-specific schema stub plus your own file-handling functions.
  - The **web search tool** — Claude runs the search itself; you just supply `max_uses` and optionally `allowed_domains` to restrict to trusted sources.
- Streaming + tools: normally the API buffers and validates each top-level JSON key before sending it (safe but chunky); `fine_grained=True` disables validation for truly real-time chunks, at the cost of needing to handle invalid JSON yourself.

# Tool Use with Claude

### Introducing tool use
- By default Claude only knows its training data — no live weather, stock prices, or current events.
- The tool use flow, in order:
  1. **Initial request** — you send a question plus instructions on how to fetch extra data.
  2. **Tool request** — Claude decides it needs more info and asks for specific data.
  3. **Data retrieval** — your server runs code to fetch that data from an API/DB.
  4. **Final response** — you send the data back; Claude answers using the original question + fresh data.
- This turns Claude from a static knowledge base into something that can work with live data.

### Project overview
- Worked example used throughout this section: teach Claude to set reminders ("remind me a week from Thursday" → "OK, I will remind you").
- Three gaps to bridge with tools, in the order they get built:
  1. **Get the current date/time** — Claude may not know the precise current time.
  2. **Add duration to a date/time** — Claude isn't reliable at date arithmetic, especially far into the future.
  3. **Set a reminder** — Claude has no native reminder mechanism at all.
- Principle: when the model has a limitation, extend it with a tool rather than fight it with prompting.

### Tool functions
- A tool function is just a plain Python function Claude can trigger when it needs extra data.
- Best practices:
  1. Use descriptive function and parameter names.
  2. Validate inputs — raise clear errors on empty/invalid params.
  3. Write meaningful error messages — ★ Claude reads errors and may retry the call with corrected input.

In [ ]:
# FUNCTION - returns current date/time in the given strftime format, validating the format string
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime()          # "2024-01-15 14:30:25"
get_current_datetime("%H:%M")   # "14:30"


### Tool schemas
- After the function, write a JSON schema so Claude knows what arguments it takes. JSON Schema is a general data-validation spec, not AI-specific.
- Three parts of a tool spec:
  1. `name` — clear, descriptive tool name.
  2. `description` — what it does, when to use it, what it returns (aim for 3-4 sentences); describe each argument in detail.
  3. `input_schema` — the actual JSON Schema for the function's arguments.
- ★ Shortcut: paste your function into Claude and ask it to generate the JSON schema for tool calling, referencing Anthropic's tool-use docs — much faster than writing it by hand.
- Naming convention: `function_name` + `function_name_schema` so schemas are easy to match to their function.
- For type safety, wrap the schema dict in `ToolParam` (from `anthropic.types`) to catch type errors early.

In [ ]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}


### Handling message blocks
- Enable tools by passing `tools=[...]` (a list of schemas) to `client.messages.create()`.
- When Claude wants to use a tool, it returns a **multi-block** assistant message, not plain text:
  1. A **text block** — human-readable explanation of what it's doing.
  2. A **ToolUse block** — `id` (for tracking), `name` (function to call), `input` (params dict), `type: "tool_use"`.
- ★ You must preserve the *entire* content list (not just the text) when appending Claude's reply to your message history, or you'll lose the tool call context: `messages.append({"role": "assistant", "content": response.content})`.
- Full tool flow, in order:
  1. Send user message + tool schema to Claude.
  2. Receive assistant message with text block + tool use block.
  3. Extract tool info and run the actual function.
  4. Send the tool result back with the complete conversation history.
  5. Receive Claude's final response.
- Helper functions (`add_user_message()`, `add_assistant_message()`) need updating to handle multi-block content, not just plain strings.

### Sending tool results
- Get the tool's requested input from the response: `response.content[1].input`, then call your function with `**` unpacking: `get_current_datetime(**response.content[1].input)`.
- Send results back in a **tool result block**, nested inside a user message:
  - `tool_use_id` — must match the originating ToolUse block's `id`.
  - `content` — the tool's output, serialized as a string.
  - `is_error` — `True` if the tool failed.
- Claude can request multiple tools in one response (e.g. two separate math questions) — each gets a unique id, and you must match ids when returning results since they can come back out of order.
- The follow-up request must still include the same `tools=[...]` schema even though you don't expect another tool call — Claude needs the schema to understand the tool references already in the conversation history.

In [ ]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[1].id,
        "content": "15:04:22",
        "is_error": False
    }]
})


### ★ Multi-turn conversations with tools
- Claude may need several tools in sequence to answer one question (e.g. "what day is 103 days from today?" needs current date, then date-math).
- Sequence when Claude needs multiple tools:
  1. User asks a question requiring chained lookups.
  2. Claude requests the first tool (e.g. `get_current_datetime`).
  3. Your server runs it and returns the result.
  4. Claude realizes it needs another tool (e.g. `add_duration_to_datetime`) and requests it.
  5. Your server runs that and returns the result.
  6. Claude now has enough information to give the final answer.
- This needs a **conversation loop** that keeps calling Claude until it stops asking for tools.
- Update helper functions to accept full `Message` objects (not just strings), and add a `text_from_message()` helper to pull out just the readable text when needed for display.

In [ ]:
# FUNCTION - accepts a string, a list of content blocks, or a full Message object
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

# FUNCTION - chat() updated to accept tool schemas and return the full message (not just text)
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    return client.messages.create(**params)

# FUNCTION - extracts and joins all text blocks from a message, for display purposes
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )


### ★ Implementing multiple turns
- Detect whether Claude wants a tool via `response.stop_reason == "tool_use"` — this is the loop's exit condition.
- The conversation loop, in order each iteration:
  1. Call `chat()` with the current messages + tool schemas.
  2. Append the assistant's reply to message history.
  3. If `stop_reason != "tool_use"`, break — Claude has a final answer.
  4. Otherwise, run every requested tool and collect `tool_result` blocks.
  5. Append the tool results as a new user message, and repeat.
- Claude can request multiple tools in a single response — filter `message.content` for blocks where `type == "tool_use"` and process each one.
- Always wrap tool execution in a try/except — a failed tool still needs a `tool_result` block sent back (with `is_error: True`) so Claude knows what happened rather than getting silence.
- Use a routing function (`run_tool(tool_name, tool_input)`) that dispatches by name to the right implementation — keeps adding new tools simple.

In [ ]:
# FUNCTION - runs the conversation loop until Claude stops requesting tools
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

# FUNCTION - executes every tool_use block in a message and builds matching tool_result blocks
def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

# FUNCTION - routes a tool call to its implementation by name
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "another_tool":
        return another_tool(**tool_input)


### Using multiple tools
- Adding a new tool once the core loop exists is a simple, repeatable pattern:
  1. Create the tool function implementation.
  2. Define its JSON schema.
  3. Add the schema to the `tools=[...]` list passed into `chat()`.
  4. Add an `elif` case for it inside `run_tool()`.
- Example that forces multiple tools in sequence: "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050" needs `add_duration_to_datetime` (to compute the date) then `set_reminder` (to save it).
- Message history for a multi-tool exchange contains, in order: user message → assistant message (text + tool use blocks) → tool result message(s) → follow-up assistant message(s).

### ★ Fine grained tool calling
- With streaming + tools, Claude sends an `InputJsonEvent` per chunk: `partial_json` (this chunk) and `snapshot` (cumulative JSON so far).
- By default, the API **buffers and validates** each top-level key-value pair before releasing it — this is why you see delayed bursts rather than smooth token-by-token streaming for tool arguments.
- `fine_grained=True` disables that validation, giving you raw, immediate chunks as Claude generates them — but you now must handle invalid/partial JSON yourself (e.g. wrap `json.loads()` in try/except).
- Use fine-grained tool calling when you need to show real-time argument generation progress or start processing partial results ASAP; stick with default validated streaming otherwise, since it's simpler and safer.

### The text edit tool
- One tool ships **built into Claude**: the text editor, giving it the ability to view, create, and edit files/directories like a real editor.
- Capabilities: view file/directory contents, view a specific line range, replace text, create new files, insert text at a line, undo recent edits.
- ★ Important distinction: the *schema* is built in, but you must still write the actual file-handling implementation — Claude only knows how to ask for the operation, not perform it.
- You do need to pass a small schema stub, version-matched to your model:
  - `claude-3-7-sonnet` → `"type": "text_editor_20250124"`
  - `claude-3-5-sonnet` → `"type": "text_editor_20241022"`
- Useful when building apps that need programmatic file editing, or embedding "AI code editor" behavior directly into your own product rather than relying on a separate IDE assistant.

### The web search tool
- Also built into Claude — unlike other tools, Claude runs the entire search itself; you only supply a schema to enable it (no implementation needed).
- Minimal schema: `{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}`. `max_uses` caps how many searches Claude can run (it may do follow-ups).
- ★ Restrict to trusted sources with `"allowed_domains": ["nih.gov"]` — valuable for domains like medical/health where you want authoritative sources only.
- Response includes several block types: text blocks, `ServerToolUseBlock` (the query used), `WebSearchToolResultBlock` / `WebSearchResultBlock` (results with titles/URLs), and citation blocks (quoted text + source).
- Best suited for: current events, specialized/recent info outside training data, fact-checking, and up-to-date research — Claude decides on its own when a search would help.
- Note: your organization must enable Web Search in the Anthropic Console privacy settings before it will work.

## Executive Summary — RAG and Agentic Search

- RAG (Retrieval Augmented Generation) solves the "document too big for one prompt" problem: chunk the document ahead of time, then only pull in the chunks relevant to a given question.
- ★ Chunking strategy determines quality — size-based (simplest, most reliable fallback, use overlap to avoid cutting mid-sentence), structure-based (best when document formatting is guaranteed, e.g. Markdown headers), sentence-based (a practical middle ground), semantic-based (most accurate, most expensive).
- Text embeddings turn a chunk into a list of numbers capturing meaning (via a model like VoyageAI's, since Anthropic doesn't provide embeddings); similarity between embeddings is measured with cosine similarity (−1 to 1, closer to 1 = more similar).
- Full RAG pipeline, in order: chunk text → embed each chunk → store embeddings in a vector DB (preprocessing, done ahead of time) → embed the user's query → search the vector DB for the closest chunks → build a final prompt with just those chunks and send to Claude.
- ★ Semantic search alone can miss exact matches (e.g. an incident ID) since it optimizes for meaning, not literal terms. **BM25** (lexical/keyword search) fixes this by weighting rare/specific terms heavily.
- Best practice: run semantic + BM25 search in parallel and merge results with **reciprocal rank fusion** (RRF) — a `Retriever` class wrapping both indexes behind a common `add_document()` / `search()` interface makes this composable and extensible to further search backends.

# RAG and Agentic Search

### Introducing Retrieval Augmented Generation
- Problem: large documents (e.g. an 800-page filing) don't fit in one prompt — and even if they technically fit, huge prompts are slower, costlier, and make Claude less effective.
- RAG's fix: chunk the document ahead of time, then at query time retrieve only the chunks relevant to the question and put just those in the prompt.
- Benefits: focuses Claude on relevant content, scales to huge/multiple documents, cheaper and faster prompts.
- Trade-offs: needs a preprocessing/chunking step, needs a relevance-search mechanism, retrieved chunks might miss needed context, and there are many possible chunking strategies to choose between.
- Use RAG when documents are large, numerous, or when cost/latency matter — not needed for content that comfortably fits in a normal prompt.

### ★ Text chunking strategies
- Bad chunking causes bad answers — e.g. a stray use of the word "bug" in a medical section could get pulled in for a software-engineering question if chunks are too coarse or mis-scoped.
- Four approaches:
  1. **Size-based** — split into equal-length strings; simplest and most reliable across any content type (including code); cuts words/sentences awkwardly unless you add overlap between chunks.
  2. **Structure-based** — split on document structure (e.g. Markdown `##` headers); cleanest, most meaningful chunks, but only works when document formatting is guaranteed.
  3. **Sentence-based** — split into sentences, then group into overlapping chunks; a practical middle ground for plain text.
  4. **Semantic-based** — group sentences by how related they are using NLP; most accurate, most computationally expensive.
- ★ Size-based chunking with overlap is the common production default — simple, reliable, and works with any content, even if not perfect.
- No single "best" strategy — the right choice depends on your documents and how much complexity you're willing to trade for chunk quality.

In [ ]:
# FUNCTION - splits text into fixed-size character chunks with overlap to preserve context at edges
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0
    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)
    return chunks

# FUNCTION - splits a Markdown document on "## " section headers
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

# FUNCTION - groups sentences into overlapping chunks of up to max_sentences_per_chunk
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start_idx = 0
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        start_idx += max_sentences_per_chunk - overlap_sentences
        if start_idx < 0:
            start_idx = 0
    return chunks


### Text embeddings
- A text embedding is a numerical representation of meaning: feed text into an embedding model, get back a long list of numbers (each between -1 and +1).
- ★ Each number is a "score" for some quality of the text, but which quality maps to which number is learned during training and isn't directly human-interpretable — thinking of dimensions as "how much about oceans" etc. is just a helpful mental model, not literal.
- Anthropic doesn't provide an embeddings model — the recommended provider is **VoyageAI** (separate account + API key, `VOYAGE_API_KEY` in `.env`).
- Semantic search = using these embeddings (rather than exact keyword matching) to find chunks related in meaning to a user's question.

In [ ]:
%pip install voyageai

from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()

# FUNCTION - generates an embedding vector for a piece of text via VoyageAI
def generate_embedding(text, model="voyage-3-large", input_type="query"):
    result = client.embed([text], model=model, input_type=input_type)
    return result.embeddings[0]


### ★ The full RAG flow
The complete pipeline, in order:
1. **Chunk the source text** into sections.
2. **Generate embeddings** for each chunk (numbers capturing meaning, then normalized to a magnitude of 1.0).
3. **Store embeddings in a vector database** — this and steps 1-2 are all preprocessing done ahead of time, before any user query arrives.
4. **Process the user's query** — embed it with the same model.
5. **Find similar embeddings** — the vector DB compares the query embedding against stored chunk embeddings using **cosine similarity** (range −1 to 1; near 1 = very similar, near 0 = unrelated, near −1 = opposite). Cosine *distance* = `1 - similarity`, so smaller distance = more similar.
6. **Build the final prompt** from the user's question plus only the best-matching chunk(s), then send to Claude.

This is the core insight of RAG: convert text to numbers, store them efficiently, then use similarity math to pull in only what's relevant when a question arrives.

### ★ Implementing the RAG flow
Concrete implementation of the five-step flow above:
1. Load the document and `chunk_by_section()` it.
2. `generate_embedding(chunks)` — batch-embed all chunks at once.
3. Create a vector store and add each `(embedding, chunk)` pair — ★ store the original chunk text alongside the embedding, since a raw embedding vector isn't useful to return to a user; you need the text back.
4. Embed the user's query the same way: `generate_embedding("...")`.
5. `store.search(user_embedding, k)` returns the top-k closest chunks with their distances (lower distance = better match).

In [ ]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)
embeddings = generate_embedding(chunks)

store = VectorIndex()
for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

user_embedding = generate_embedding("What did the software engineering dept do last year?")
results = store.search(user_embedding, 2)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")


### ★ BM25 lexical search
- Semantic search alone can miss exact-term lookups (e.g. an incident ID like "INC-2023-Q4-011") because it optimizes for conceptual similarity, not literal matches.
- Fix: run semantic search and lexical search (BM25) in parallel, then merge — get both meaning-based and exact-match recall.
- How BM25 scores a query, in order:
  1. **Tokenize** the query into individual terms.
  2. **Count term frequency** across all documents.
  3. **Weight rarer terms higher** — common words (e.g. "a") get low importance, rare/specific terms (e.g. an incident ID) get high importance.
  4. **Return documents** containing more instances of the higher-weighted terms.
- Best for technical terms, IDs, and specific phrases that semantic search tends to under-rank.

In [ ]:
chunks = chunk_by_section(text)

store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

results = store.search("What happened with INC-2023-Q4-011?", 3)
for doc, distance in results:
    print(distance, "\n", doc["content"][:200], "\n----\n")


### ★ A Multi-Index RAG pipeline
- `VectorIndex` (semantic) and `BM25Index` (lexical) share the same API — `add_document()` / `search()` — so they can be wrapped in one `Retriever` class that queries both and merges results.
- Merging uses **reciprocal rank fusion (RRF)**, since the two indexes score results on different, incompatible scales:
  1. Get each index's ranked results for the query.
  2. For every document, sum `1 / (k + rank_i(d))` across each index's ranking of it (k is a constant, often 60).
  3. Sort documents by this combined RRF score — documents that rank well in *both* indexes rise to the top.
- Example: a doc ranked 1st in vector search and 2nd in BM25 will typically outscore one that's 3rd/1st, rewarding consistent relevance across both methods.
- Extensible by design: any new search method that implements the same `add_document()` / `search()` interface (keyword index, graph search, domain-specific index) can be added to the `Retriever` with no changes to the fusion logic.

In [ ]:
class Retriever:
    # FUNCTION - wraps multiple search indexes (e.g. VectorIndex, BM25Index) behind one interface
    def __init__(self, *indexes):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    # FUNCTION - adds a document to every wrapped index
    def add_document(self, document):
        for index in self._indexes:
            index.add_document(document)

    # FUNCTION - queries every index and merges results via reciprocal rank fusion
    def search(self, query_text, k=1, k_rrf=60):
        all_results = [index.search(query_text, k) for index in self._indexes]
        scores = {}
        for results in all_results:
            for rank, (doc, _) in enumerate(results):
                key = doc["content"]
                scores[key] = scores.get(key, 0) + 1.0 / (k_rrf + rank + 1)
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:k]


## Executive Summary — Features of Claude

- **Extended thinking** gives Claude a visible "scratch pad" before its final answer — better reasoning and transparency, at the cost of higher latency/tokens. ★ Decide whether to use it based on evals: optimize your prompt first, only add thinking if accuracy still falls short. Not compatible with prefilling or `temperature`.
- **Image support**: send images as base64 or URL image blocks alongside text; same prompt-engineering techniques that improve text (specific guidelines, step-by-step methodology, one-shot examples) dramatically improve visual analysis accuracy too — simple questions like "how many X" often need a structured methodology to get right.
- **PDF support**: nearly identical to image handling — swap `"type": "image"` → `"type": "document"`, media type → `"application/pdf"`. Claude can read text, embedded images/charts, and table structure, not just raw text.
- ★ **Citations**: add `"title"` + `"citations": {"enabled": True}` to a document block to get per-claim `cited_text`, `document_index/title`, and page (or character, for plain text) locations back — turns Claude from a black box into a transparent, verifiable research assistant.
- ★ **Prompt caching**: cache breakpoints (`cache_control: {"type": "ephemeral"}`) save preprocessing work for reuse within a 1-hour window — cheaper/faster on repeated large system prompts, tool schemas, or long documents. Minimum 1024 tokens to be eligible; even a single changed character invalidates the cache from that point on.
- **Code execution + Files API**: upload files ahead of time (get a file ID back) instead of inlining base64 data, then let Claude write and run Python in an isolated, network-less Docker container — ideal for data analysis, since the Files API is the only way to get data in/out of that sandboxed container.

# Features of Claude

### Extended thinking
- Extended thinking gives Claude a "scratch paper" — a visible reasoning block before the final answer — improving quality and transparency on hard problems, at the cost of higher latency and token cost.
- ★ Decision rule: don't reach for thinking by default. Run your evals without it first; only enable it if accuracy still isn't meeting requirements after you've already optimized the prompt.
- Not compatible with some other features — notably message prefilling and `temperature`.
- Responses include a cryptographic signature on the thinking block, preventing tampering with Claude's reasoning (a safety measure).
- Sometimes you get a **redacted thinking** block instead of readable text — this happens when internal safety systems flag the reasoning; the content is still passed back encrypted so context isn't lost across turns.
- Enable via `thinking=True` + `thinking_budget` (minimum 1024 tokens; `max_tokens` must exceed the thinking budget).

In [ ]:
def chat(
    messages, system=None, temperature=1.0, stop_sequences=[],
    tools=None, thinking=False, thinking_budget=1024
):
    params = {"model": model, "max_tokens": 2000, "messages": messages}
    if thinking:
        params["thinking"] = {"type": "enabled", "budget": thinking_budget}
    return client.messages.create(**params)

chat(messages, thinking=True)


### Image support
- Limits to know: up to 100 images per request, max 5MB/image, max 8000px (single image) or 2000px (multiple images), base64 or URL input. Token cost ≈ `(width_px * height_px) / 750`.
- Send an image as a block alongside a text block in the same user message (see code below).
- ★ The same prompt-engineering techniques that improve text responses dramatically improve image analysis too — a plain question ("how many marbles?") is unreliable; providing a step-by-step counting methodology or a one-shot example with a known answer meaningfully raises accuracy.
- Real-world example: satellite-image fire-risk scoring — a prompt broken into ordered steps (identify residence → check tree overhang → assess fire vulnerability → check defensible space → assign a 1-4 rating) produces far more consistent results than asking for "a risk score" directly.

In [ ]:
with open("image.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    {
        "type": "image",
        "source": {"type": "base64", "media_type": "image/png", "data": image_bytes}
    },
    {"type": "text", "text": "What do you see in this image?"}
])


### PDF support
- Nearly identical to image handling — Claude can read text, embedded images/charts, tables, and document structure from a PDF, not just extracted text.
- Changes needed vs. the image code: file extension `.pdf`, variable name → `file_bytes` (for clarity), block `"type"` → `"document"`, `media_type` → `"application/pdf"`.

In [ ]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    {
        "type": "document",
        "source": {"type": "base64", "media_type": "application/pdf", "data": file_bytes},
    },
    {"type": "text", "text": "Summarize the document in one sentence"},
])


### ★ Citations
- Add `"title"` and `"citations": {"enabled": True}` to a document block to make Claude track and report exactly where in the source it found each claim.
- Each citation includes: `cited_text` (exact source snippet), `document_index`, `document_title`, `start_page_number` / `end_page_number` (or character positions for plain-text sources instead of page numbers).
- Turns Claude from a "black box" into a transparent, verifiable research assistant — use whenever users need to verify facts, you're working with authoritative documents, or source transparency matters.
- Works with PDFs and plain text documents alike (just swap `source.type` to `"text"` / `media_type` to `"text/plain"`).

In [ ]:
{
    "type": "document",
    "source": {"type": "base64", "media_type": "application/pdf", "data": file_bytes},
    "title": "earth.pdf",
    "citations": {"enabled": True}
}


### ★ Prompt caching
- Claude normally tokenizes, embeds, and contextualizes your input, then throws all that preprocessing work away after responding — wasteful if you resend similar content.
- Prompt caching stores that preprocessing so a follow-up request with identical content can reuse it instead of redoing the work.
- Benefits: faster responses and lower cost on cached portions. Limits: cache lives only **1 hour**, and it only helps when you're repeatedly sending the *same* content.
- Best fits: document-analysis workflows (many questions about one big doc) and iterative editing (base content stable, small edits on top).

### ★ Rules of prompt caching
- Caching is **not automatic** — you must manually add a cache breakpoint to a block, using the longhand block form with `"cache_control": {"type": "ephemeral"}` (the shorthand string form has nowhere to put this field).
- A breakpoint caches everything before and including it; the cache is only reused if content up to that breakpoint is byte-for-byte identical on the next call — even adding one word invalidates it.
- Breakpoints can span multiple prior messages, and can be placed on text blocks, system prompts, tool definitions, image blocks, or tool use/result blocks.
- ★ System prompts and tool definitions are prime caching candidates since they rarely change between requests — usually where you get the most benefit.
- Processing order behind the scenes: **tools → system prompt → messages**. Up to 4 cache breakpoints allowed total, so you can cache tools and separately cache part of the conversation history.
- Minimum cacheable size: 1024 tokens (summed across everything up to the breakpoint) — trivially short prompts won't qualify.

### ★ Prompt caching in action
- To cache tool schemas: copy the tools list and the last tool, add `cache_control` to that copy (avoids mutating your original definitions).
- To cache a system prompt: convert it from a plain string into a list with one text block carrying `cache_control`.
- Response usage fields tell you what happened: `cache_creation_input_tokens` (first request, writing to cache) vs. `cache_read_input_tokens` (follow-up, reading from cache) — a changed system prompt with unchanged tools shows a partial read (tools) + partial write (new system prompt), so you only pay for what actually changed.

In [ ]:
# Caching tool schemas (copy first, don't mutate the original list)
if tools:
    tools_clone = tools.copy()
    last_tool = tools_clone[-1].copy()
    last_tool["cache_control"] = {"type": "ephemeral"}
    tools_clone[-1] = last_tool
    params["tools"] = tools_clone

# Caching a system prompt
if system:
    params["system"] = [
        {"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}
    ]


### Code execution and the Files API
- **Files API**: upload a file once (image, PDF, text, CSV, etc.) and get back a file ID to reference in later messages, instead of re-encoding base64 data every time — best for files reused across requests or files too large to comfortably inline.
- **Code execution tool**: a server-side, schema-only tool (`{"type": "code_execution_20250522", "name": "code_execution"}`) — Claude writes and runs Python itself in an isolated Docker container with **no network access**, potentially executing code multiple times in one conversation.
- ★ Because the container has no network access, the **Files API is the only way to move data in and out** of it — upload your input file, reference it via a `container_upload` block, and download any generated outputs (plots, reports) the same way.
- Response includes text blocks (explanation), server tool use blocks (the actual code run), and code execution result blocks (output) — look for `type: "code_execution_output"` blocks to grab file IDs for generated content.
- Use cases beyond data analysis: image processing, document parsing/transformation, math/modeling, custom report generation — anywhere you want to delegate a genuinely computational task to Claude.

In [ ]:
file_metadata = upload('streaming.csv')

messages = []
add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": "Run a detailed analysis to determine major drivers of churn. "
                    "Your final output should include at least one detailed plot summarizing your findings."
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250522", "name": "code_execution"}])

# Later, download anything Claude generated:
download_file("file_id_from_response")


## Executive Summary — Model Context Protocol (MCP)

- ★ MCP is a protocol that shifts the burden of *writing and maintaining* tool integrations away from your own server to dedicated **MCP servers** — it's a different concern from tool use itself: someone else has already authored the schemas and functions, packaged behind a standard interface.
- Architecture: your app's **MCP client** talks to one or more **MCP servers** using a transport-agnostic connection (commonly stdio on the same machine, or HTTP/WebSockets); message types include `ListToolsRequest/Result` and `CallToolRequest/Result`.
- Full request flow, in order: get tools from the MCP client → send user question + tools to Claude → Claude requests a tool → client asks the MCP server to run it (`CallToolRequest`) → result flows back → sent to Claude as a follow-up → Claude gives the final answer.
- The official Python SDK (`FastMCP`) turns tool/resource/prompt definitions into simple decorated functions with type hints — no manual JSON schema writing required.
- MCP servers expose three kinds of things: **tools** (actions, e.g. edit a document), **resources** (data to fetch, like a GET request — direct or templated URIs), and **prompts** (pre-built, tested message templates parameterized by arguments).
- The MCP Inspector (`mcp dev mcp_server.py`) lets you test tools/resources/prompts in isolation in the browser before wiring up a full client.
- In real projects you typically build *either* a client or a server, not both — this course builds both purely to show how they communicate.

# Model Context Protocol (MCP)

### ★ Introducing MCP
- MCP is a communication layer providing Claude context and tools without you writing all the integration code yourself.
- Example: a GitHub chatbot needs many tools (repos, PRs, issues, projects...) — without MCP you'd author every schema + function yourself.
- MCP servers wrap that functionality: they package pre-built tools for a service (e.g. GitHub) into a reusable component any application can connect to.
- Common misconceptions clarified:
  - **MCP is not "just tool use."** Tool use is the mechanism; MCP is about *who writes and maintains* the tools — with MCP, someone else already has.
  - **MCP differs from calling an API directly** because the server already defines the tool schemas/functions for you; direct API integration means you author them yourself.
- Anyone can author an MCP server — often service providers release their own official implementations (e.g. an AWS MCP server for AWS services).

### MCP clients
- The MCP client is the bridge between your server and an MCP server — it handles protocol/message-passing details for you.
- **Transport agnostic**: client and server can communicate via stdio (same machine, most common), HTTP, WebSockets, or other protocols.
- Core message types: `ListToolsRequest/Result` (what tools are available) and `CallToolRequest/Result` (run a tool, get its result).
- Full communication flow for a query like "what repositories do I have?", in order:
  1. User submits a query to your server.
  2. Your server asks the MCP client for available tools (`ListToolsRequest` → `ListToolsResult`).
  3. Your server sends the user's question + those tools to Claude.
  4. Claude decides it needs a tool and responds with a tool use request.
  5. Your server asks the MCP client to run that tool (`CallToolRequest` → server calls e.g. GitHub → `CallToolResult`).
  6. Your server sends the tool result back to Claude.
  7. Claude responds with the final answer, which your server returns to the user.

### Project setup
- Hands-on project: a CLI chatbot with an MCP client (handles user interaction) and a custom MCP server (manages in-memory documents) — two tools: read a document, edit a document.
- ★ Real-world note: normally you build *either* a client or a server, not both — this project builds both purely for learning how they talk to each other.
- Setup, in order: download and extract `cli_project.zip` → add your Anthropic API key to `.env` → install deps with UV (recommended) or pip → run `uv run main.py` (or `python main.py`) → verify with a simple question like "what's 1+1?".

### Defining tools with MCP
- The Python MCP SDK (`FastMCP`) replaces manual JSON schema writing with decorators + type hints — schemas are auto-generated from your function signature.
- Server init is one line: `mcp = FastMCP("DocumentMCP", log_level="ERROR")`. Example server stores documents in a plain in-memory dict.
- Define a tool with `@mcp.tool(name=..., description=...)` over a function whose parameters use `Field(description=...)` (from Pydantic) for per-argument docs — the decorator generates the schema Claude needs automatically.
- Both example tools (`read_doc_contents`, `edit_document`) raise a `ValueError` with a descriptive message when given an invalid `doc_id` — same "validate + clear error" best practice as manually-defined tools, so Claude can react sensibly.
- Benefits over hand-written schemas: auto-generated JSON schema, Pydantic-based parameter validation, less boilerplate, IDE/type support.

In [ ]:
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("DocumentMCP", log_level="ERROR")

docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
}

# FUNCTION - reads a document's contents by id, raising an error if not found
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(doc_id: str = Field(description="Id of the document to read")):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

# FUNCTION - does an in-place find-and-replace on a document's contents
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)


### The server inspector
- The Python SDK ships a browser-based **MCP Inspector** for testing a server without wiring up a full client/app.
- Usage, in order:
  1. Start it: `mcp dev mcp_server.py` (runs on port 6277, gives you a local URL).
  2. Click **Connect** to start your server from the inspector.
  3. Go to the **Tools** section → **List Tools** to see everything available.
  4. Select a tool, fill in parameters, click **Run Tool** to see results.
  5. Chain calls to verify behavior (e.g. edit a document, then immediately read it back to confirm the change stuck).
- Makes for a fast dev loop: edit server code → test tools directly in the inspector → debug in isolation, no need to connect to Claude or a full app just to sanity-check a tool.

### Implementing a client
- Two components make up the client side: your own **MCPClient** wrapper class (handles setup/cleanup) and the SDK's **ClientSession** (the actual connection).
- Two core methods to implement: `list_tools()` and `call_tool()` — both just forward to the underlying session and return its result.
- Real projects build a client *or* a server, not both — again, this course builds both for teaching purposes.
- End-to-end test flow, in order: get tools via the client → send tools + question to Claude → Claude picks a tool → client calls it on the server → result goes back to Claude → Claude answers the user.

In [ ]:
# FUNCTION - fetches the list of tools the MCP server exposes
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools

# FUNCTION - runs a named tool on the MCP server with the given input
async def call_tool(self, tool_name: str, tool_input: dict) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)

# Quick manual test
async with MCPClient(command="uv", args=["run", "mcp_server.py"]) as client:
    result = await client.list_tools()
    print(result)


### Defining resources
- Resources expose **data to fetch** (like an HTTP GET) as opposed to tools, which perform actions — used e.g. for an "@document_name" mention/autocomplete feature.
- Two resource types:
  1. **Direct resources** — a fixed, unparameterized URI, e.g. `docs://documents` (list all docs).
  2. **Templated resources** — a URI with parameters, e.g. `docs://documents/{doc_id}`; the SDK auto-parses the URI and passes params as keyword args to your function.
- Define with `@mcp.resource(uri, mime_type=...)`; the SDK auto-serializes your return value (JSON, plain text, etc. — no manual conversion needed).
- Test resources the same way as tools, via the MCP Inspector's **Resources** / **Resource Templates** sections.

In [ ]:
# FUNCTION - direct resource: lists all available document ids
@mcp.resource("docs://documents", mime_type="application/json")
def list_docs() -> list[str]:
    return list(docs.keys())

# FUNCTION - templated resource: fetches one document's contents by id
@mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


### Accessing resources
- Client-side counterpart to defining resources: implement `read_resource(uri)`, which calls the session and pulls the first item out of `result.contents`.
- Use the returned `mimeType` to decide how to parse the payload — `application/json` gets `json.loads()`'d, everything else is returned as plain text.
- ★ Advantage over tools: the document content goes straight into the prompt (e.g. via an "@report.pdf" mention), skipping a tool-call round-trip entirely — faster and more efficient when you just need to inject data, not perform an action.

In [ ]:
import json
from pydantic import AnyUrl

# FUNCTION - fetches a resource by URI and parses it based on its MIME type
async def read_resource(self, uri: str):
    result = await self.session().read_resource(AnyUrl(uri))
    resource = result.contents[0]

    if isinstance(resource, types.TextResourceContents):
        if resource.mimeType == "application/json":
            return json.loads(resource.text)
        return resource.text


### Defining prompts
- MCP prompts are pre-built, well-tested message templates a server author provides, so clients get expert-crafted instructions instead of writing ad hoc prompts themselves.
- ★ The value proposition: a user *could* type "convert report.pdf to markdown" themselves, but a carefully tested prompt (with explicit formatting/structure/output rules) reliably gets better, more consistent results.
- Define with `@mcp.prompt(name=..., description=...)` over a function that returns a list of `base.Message` objects (e.g. `base.UserMessage(prompt)`).
- Best practices: focus prompts on tasks central to your server's purpose, write detailed/specific instructions (not vague ones), test thoroughly across inputs, and design them to work well with your server's own tools.

In [ ]:
from mcp.server.fastmcp import base

# FUNCTION - a pre-built prompt template that reformats a document into Markdown
@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format."
)
def format_document(doc_id: str = Field(description="Id of the document to format")) -> list[base.Message]:
    prompt = f"""
    Your goal is to reformat a document to be written with markdown syntax.
    The id of the document you need to reformat is: {doc_id}
    Add in headers, bullet points, tables, etc as necessary.
    Use the 'edit_document' tool to edit the document.
    """
    return [base.UserMessage(prompt)]


### Prompts in the client
- Client-side methods: `list_prompts()` (mirrors `list_tools()`) and `get_prompt(prompt_name, args)`, which returns the fully interpolated message list ready to send to Claude.
- Arguments passed to `get_prompt()` are forwarded as keyword arguments to the server-side prompt function — this is how dynamic values (like a `doc_id`) get substituted into the template.
- CLI workflow, in order: user selects a prompt (e.g. via a `/` command) → system asks for required arguments (e.g. which document) → the fully interpolated prompt is sent to Claude → Claude may use tools to complete the task.

In [ ]:
# FUNCTION - fetches all prompts the MCP server exposes
async def list_prompts(self) -> list[types.Prompt]:
    result = await self.session().list_prompts()
    return result.prompts

# FUNCTION - fetches one prompt with arguments interpolated into it
async def get_prompt(self, prompt_name, args: dict[str, str]):
    result = await self.session().get_prompt(prompt_name, args)
    return result.messages


### MCP review
- Recap lesson (video walkthrough of the completed CLI project) tying together everything built in this section: the FastMCP server (tools + resources + prompts), the custom MCPClient wrapper, and the end-to-end flow between user, client, MCP server, and Claude.